# Sprint 3 · Qwen и письма
T4 GPU. Входной архив: прежние HC3/CoAT + скоры GPT-Neo + разрешённые LLMGen/Nazario. Исходные тексты не сохраняются в Git. Веса Qwen не дообучаются.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
def run(cmd):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end='', flush=True)
    if p.wait(): raise RuntimeError(cmd)
ROOT = Path('/content/DW-1-sprint3')
if not ROOT.exists():
    run(['git','clone','--branch','w1/ru-detector-email-scores','https://github.com/spotyyyi123-ux/DW-1.git',str(ROOT)])
os.chdir(ROOT)
run([sys.executable,'-m','pip','install','-q','-r','scripts/external/requirements.txt'])
os.environ['PYTHONPATH'] = str(ROOT/'src')
sys.path.insert(0,str(ROOT/'src'))
import torch
assert torch.cuda.is_available(), 'Выберите T4 GPU'
print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
from google.colab import files
from zipfile import ZipFile
uploaded = files.upload()
assert len(uploaded) == 1
with ZipFile(next(iter(uploaded))) as z:
    allowed={'llm_benchmark.jsonl','llm_benchmark_emails.jsonl','gptneo_scores.parquet'}
    assert set(z.namelist())==allowed
    for name in allowed:
        dst = ROOT / ('data/runs/sprint3' if name.endswith('.parquet') else 'data/sprint2') / name
        dst.parent.mkdir(parents=True,exist_ok=True)
        dst.write_bytes(z.read(name))


In [ ]:
revision = '060db6499f32faf8b98477b0a26969ef7d8b9987'
for benchmark, output in [('llm_benchmark.jsonl','qwen_scores.parquet'), ('llm_benchmark_emails.jsonl','email_scores.parquet')]:
    run([sys.executable,'scripts/run_qwen_local.py','--device','cuda','--model','Qwen/Qwen2.5-0.5B','--revision',revision,'--max-tokens','512','--benchmark','data/sprint2/'+benchmark,'--out','data/runs/sprint3/'+output])


In [ ]:
run([sys.executable,'scripts/evaluate_sprint3_llm.py'])


In [ ]:
from zipfile import ZIP_DEFLATED
archive = Path('/content/dw1-sprint3-results.zip')
with ZipFile(archive,'w',ZIP_DEFLATED) as z:
    for p in sorted(Path('data/runs/sprint3').iterdir()):
        if p.is_file(): z.write(p,arcname=p.name)
files.download(str(archive))
